# Day 6 — Modules & Project Structure

> ⚠️ **Why this matters.** Your english-helper is now 5 files in one folder. Add 5 more and it's chaos. Modules and packages are how you keep growing without drowning in spaghetti. By the end of today your project will look like real software, not a homework folder.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jakkzz/prince-curriculum/blob/main/phase-1-python-cli/lessons/06-modules.ipynb)

## What you'll do today

**Time:** 90 min lesson + 60 min refactor + 45 min quiz.

By the end:

- [ ] You know the difference between a module, a package, and a script
- [ ] You can import like a professional (`from x import y` vs `import x`)
- [ ] You know what `__name__ == "__main__"` actually does
- [ ] Your english-helper is reorganized as a real package under `src/`

## The mental model

- **Module** = a `.py` file (`dictionary.py` is a module called `dictionary`)
- **Package** = a folder of modules with an `__init__.py` file (or empty for namespace packages)
- **Script** = a file you run with `python file.py`

A file can be all three. The same `vocab_app.py` is a script when you run it directly, a module if someone imports it.

> 💡 **In the wild:** Every Python project at scale (Django, Flask, FastAPI, Pandas) is organized as a package with modules. You're learning the layout that all real software uses.

## 1. Importing — three forms

In [ ]:
# Form 1: import the module
import json
data = json.loads('{"a": 1}')

# Form 2: import specific names
from json import loads, dumps
data = loads('{"a": 1}')

# Form 3: aliased import (use sparingly)
import numpy as np   # convention for numpy
import pandas as pd  # convention for pandas
# Don't invent aliases that aren't conventional — they confuse readers

> 🎯 **Rule of thumb:** Use `import x` for clarity (`json.loads()` reads better than `loads()`). Use `from x import y` for very common things or to avoid a long name (`from datetime import datetime` so you can say `datetime.now()`).

## 2. Your own modules

In [ ]:
# In english-helper/word_utils.py:
def slugify(text: str) -> str:
    return text.lower().strip().replace(' ', '-')

# In english-helper/vocab_app.py:
from word_utils import slugify
print(slugify('The Big Word'))   # the-big-word

Python finds modules by searching `sys.path` — your project root, the standard library, then installed packages.

**For the english-helper refactor today**, every file you've written becomes a module others can import:
- `dictionary.py` → import `lookup`, `add_word`, `WordNotFoundError`
- `vocab_app.py` → uses `from dictionary import lookup`
- A new `cli.py` → imports from everything else

## 3. `__name__ == "__main__"`

In [ ]:
# In any module:
def main():
    print('Running as a script')

if __name__ == '__main__':
    main()

When a file is **run directly**, Python sets `__name__ = '__main__'`. When **imported**, `__name__` is the module name.

**What this lets you do:** put a demo at the bottom of every module — runs when you do `python my_module.py`, stays out of the way when someone imports.

From now on **every script you write ends with this pattern**. Even tiny ones. It costs you 2 lines and saves a future bug.

## 4. Packages — folders with `__init__.py`

Folder layout for a proper Python package:

```
english-helper/
├── pyproject.toml
├── src/
│   └── english_helper/         ← package root, has __init__.py
│       ├── __init__.py
│       ├── dictionary.py
│       ├── storage.py
│       ├── quiz.py
│       └── cli.py
└── tests/
```

Note the underscore — Python imports use `english_helper`, not `english-helper`. Folder names can have dashes; module names can't.

`__init__.py` can be empty. Its presence is what makes the folder a package. You can also use it to control what's exported.

**Imports look like this from anywhere in the project:**

```python
from english_helper.dictionary import lookup
from english_helper import storage
```

## End-of-day mini-project — refactor english-helper into a package

> 🎯 **Today's piece:** restructure the flat folder into a real Python package layout. No new features — just structural.

### Steps

1. Edit `pyproject.toml` to use src layout:
   ```toml
   [tool.hatch.build.targets.wheel]
   packages = ["src/english_helper"]
   ```
2. Make the folders:
   ```bash
   cd ~/prince/scratch/english-helper
   mkdir -p src/english_helper tests
   touch src/english_helper/__init__.py
   ```
3. Move and rename your files:
   ```bash
   git mv dictionary.py src/english_helper/dictionary.py
   git mv vocab_app.py src/english_helper/cli.py
   # storage logic from cli.py → src/english_helper/storage.py (split it out)
   ```
4. Update all the imports.
5. Run: `uv sync` then `uv run python -m english_helper.cli` — it should work just like before.
6. mypy + ruff still pass.

### Stretch
- Add a `__main__.py` so `uv run python -m english_helper` runs the CLI.
- Export a clean public API in `__init__.py`: `from .dictionary import lookup`.
- Add an entry point in pyproject.toml so `uv run english-helper` works.

## Connect to the project

> 🎯 **Connects to the project:** Tomorrow you start hitting real APIs with `requests`. That'll add a new module (`api.py`). Without today's package structure, that file would get tangled with everything else. With it, you just drop in a new file and import from it.

## Self-check

<details>
<summary>1. What's the difference between a module and a package?</summary>

Module = single `.py` file. Package = folder of modules with `__init__.py`. A package can contain modules and sub-packages.
</details>

<details>
<summary>2. When does Python set <code>__name__ = '__main__'</code>?</summary>

When the file is run directly (`python file.py` or `uv run python file.py`). When imported, `__name__` is the module's name (e.g., `english_helper.dictionary`).
</details>

<details>
<summary>3. Why <code>english_helper</code> not <code>english-helper</code> for the package name?</summary>

Python identifiers can't contain hyphens — `import english-helper` would be parsed as subtraction. Use underscores in module/package names. The git repo or PyPI distribution name can have dashes; the import name must use underscores.
</details>

**Quiz:** [06-modules-quiz.ipynb](06-modules-quiz.ipynb)